# GF4 Codebook-Regularity Sweep (Colab)

Measures how much **WikiText-2 perplexity** GF4 loses when its Gaussian-quantile codebook is **regularized** toward coarser fixed-point / power-of-two levels (which would synthesize to a cheaper, more shift-friendly hardware decode).

**Method (faithful GF4 regime):** each targeted `nn.Linear` input is rotated by a fixed randomized blockwise Hadamard (block=32), the same rotation applied once to the weight columns (output preserved), then the rotated activation is GF4-quantized with the codebook under test. Weights stay fp16 (A-only), isolating the **activation codebook**.

**Two axes:** codebook precision `Q1.B` (round levels to `k/2^B`) and block-scale precision (`fp16` vs `e4m3`, GF4's real weight-scale format). The codebook quantizes *shape* after per-block RMS normalization; the scale carries *range* — which is why coarse codebooks (Q1.4) stay near-lossless.

Set the GPU runtime (**Runtime → Change runtime type → GPU**), then run all cells.

In [ ]:
!pip install -q transformers datasets accelerate scipy

In [ ]:
import math, csv
import numpy as np
import torch, torch.nn as nn
from scipy.linalg import hadamard as scipy_hadamard

GF4_LEVEL = np.array([0.0, 0.0796082, 0.1737177, 0.2828685,
                      0.3952704, 0.5250730, 0.6961928, 1.0], dtype=np.float64)
BLOCK = 32
CLIP_RATIO = 2.5

# ---- codebook family ------------------------------------------------------
def snap_q1b(levels, B):
    g = np.round(levels * (2 ** B)) / (2 ** B); g[0], g[-1] = 0.0, 1.0; return g
def snap_pow2(levels):
    out = levels.copy()
    for i, v in enumerate(levels):
        out[i] = (2.0 ** round(math.log2(v))) if v > 0 else 0.0
    out[-1] = 1.0; return out
def build_codebooks(which="all"):
    b = {"exact": GF4_LEVEL.copy()}
    for B in (6, 5, 4, 3): b[f"q1b{B}"] = snap_q1b(GF4_LEVEL, B)
    b["pow2"] = snap_pow2(GF4_LEVEL)
    if which != "all": b = {k: v for k, v in b.items() if k in which.split(",")}
    return b

# ---- GF4 block scale (E4M3, GF4's deployed weight-scale format) ------------
def quant_e4m3(scale):
    s = scale.clamp(min=2.0 ** -9, max=448.0)
    e = torch.floor(torch.log2(s))
    m = torch.round((s / torch.exp2(e) - 1.0) * 8.0) / 8.0
    return (1.0 + m) * torch.exp2(e)
_SCALE_MODE = "fp16"

# ---- active codebook ------------------------------------------------------
_LEVELS = _THR = None
def set_codebook(levels_np, device):
    global _LEVELS, _THR
    lv = np.sort(levels_np).astype(np.float32)
    thr = np.array([(lv[i] + lv[i + 1]) / 2 for i in range(len(lv) - 1)], dtype=np.float32)
    _LEVELS = torch.tensor(lv, device=device); _THR = torch.tensor(thr, device=device)

# ---- blockwise randomized Hadamard ----------------------------------------
_H = None
def hmat(device, dtype):
    global _H
    if _H is None or _H.device != device:
        _H = torch.tensor(scipy_hadamard(BLOCK).astype(np.float32) / math.sqrt(BLOCK), device=device)
    return _H.to(dtype)
def rotate(x, signs):
    F_ = x.shape[-1]; H = hmat(x.device, x.dtype)
    xr = (x * signs).reshape(*x.shape[:-1], F_ // BLOCK, BLOCK) @ H
    return xr.reshape(*x.shape[:-1], F_)

# ---- GF4 activation quant (active codebook + scale mode) ------------------
def gf4_quant(x):
    shp = x.shape
    xb = x.reshape(-1, BLOCK).float()
    rms = xb.pow(2).mean(-1, keepdim=True).add(1e-12).sqrt()
    scale = rms * CLIP_RATIO
    if _SCALE_MODE == "e4m3": scale = quant_e4m3(scale)
    xn = xb / scale
    mag = xn.abs().clamp(0.0, 1.0)
    idx = torch.bucketize(mag, _THR)
    deq = _LEVELS[idx] * torch.sign(xn) * scale
    return deq.reshape(shp).to(x.dtype)

# ---- hooks: rotate weights once, quantize rotated input per forward -------
def install_hooks(model, seed=0):
    g = torch.Generator().manual_seed(seed); n = 0
    for name, m in model.named_modules():
        if isinstance(m, nn.Linear) and m.in_features % BLOCK == 0 and "lm_head" not in name:
            dev = m.weight.device
            signs = (torch.randint(0, 2, (m.in_features,), generator=g).float() * 2 - 1).to(dev)
            m.register_buffer("_gf4_signs", signs, persistent=False)
            with torch.no_grad():
                m.weight.data.copy_(rotate(m.weight.data.float(), signs).to(m.weight.dtype))
            def pre_hook(mod, inp):
                x = inp[0]
                return (gf4_quant(rotate(x.float(), mod._gf4_signs)).to(x.dtype),) + inp[1:]
            m.register_forward_pre_hook(pre_hook); n += 1
    return n

# ---- WikiText-2 perplexity ------------------------------------------------
def load_windows(tok, seqlen, n):
    from datasets import load_dataset
    try:
        ds = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")
    except Exception:
        ds = load_dataset("wikitext", "wikitext-2-raw-v1", split="test", trust_remote_code=True)
    ids = tok("\n\n".join(ds["text"]), return_tensors="pt").input_ids[0]
    k = min(n, ids.shape[0] // seqlen)
    return [ids[i * seqlen:(i + 1) * seqlen] for i in range(k)]

@torch.no_grad()
def perplexity(model, windows):
    dev = next(model.parameters()).device
    nll = ntok = 0
    for w in windows:
        w = w.to(dev)
        out = model(w.unsqueeze(0), labels=w.unsqueeze(0))
        nll += out.loss.item() * (w.numel() - 1); ntok += w.numel() - 1
    return math.exp(nll / ntok)
print("functions defined.")

In [ ]:
# ======================= CONFIG =======================
MODEL        = "facebook/opt-1.3b"   # e.g. "meta-llama/Llama-2-7b-hf"
EVAL_WINDOWS = 20
SEQLEN       = 2048
SCALE_MODES  = ["fp16", "e4m3"]      # e4m3 = GF4's real block-scale format
CODEBOOKS    = "all"                 # or e.g. "exact,q1b4,pow2"
LOAD_8BIT    = False                 # only to FIT a big model (quantizes base weights)
SEED         = 0
# ======================================================
from transformers import AutoModelForCausalLM, AutoTokenizer
print(f"Loading {MODEL} ...")
tok = AutoTokenizer.from_pretrained(MODEL, use_fast=False)
kw = dict(device_map="auto", torch_dtype=torch.float16)
if LOAD_8BIT: kw.update(load_in_8bit=True); kw.pop("torch_dtype")
model = AutoModelForCausalLM.from_pretrained(MODEL, **kw).eval()
windows = load_windows(tok, SEQLEN, EVAL_WINDOWS)
print(f"{len(windows)} windows x {SEQLEN} tok")

ppl_fp16 = perplexity(model, windows)
print(f"baseline fp16 PPL: {ppl_fp16:.4f}")
nh = install_hooks(model, SEED)
print(f"hooked {nh} linears (A-only GF4, blockwise Hadamard)\n")

books = build_codebooks(CODEBOOKS)
dev = next(model.parameters()).device
rows, exact = [], {}
for sm in SCALE_MODES:
    _SCALE_MODE = sm
    for name, levels in books.items():
        set_codebook(levels, dev)
        ppl = perplexity(model, windows)
        if name == "exact": exact[sm] = ppl
        rows.append((sm, name, ppl, ppl - ppl_fp16, ppl - exact.get(sm, float("nan"))))
        print(f"  scale={sm:5s} {name:6s} PPL {ppl:8.4f}  dPPL(fp16) {ppl-ppl_fp16:+7.3f}  dPPL(exact) {ppl-exact.get(sm,float('nan')):+7.3f}")

with open("gf4_regularity_results.csv", "w", newline="") as f:
    w = csv.writer(f); w.writerow(["scale_mode","codebook","ppl","dppl_fp16","dppl_exact"])
    for sm, name, ppl, d1, d2 in rows: w.writerow([sm, name, f"{ppl:.4f}", f"{d1:.4f}", f"{d2:.4f}"])
print(f"\nbaseline fp16 {ppl_fp16:.4f}; wrote gf4_regularity_results.csv")

In [ ]:
# Pareto: dPPL (vs exact GF4) vs decode-ROM area savings.
# Decode-area numbers are from local RTL synthesis (Yosys/Nangate45), embedded
# here since yosys is not on Colab: exact=15.162, Q1.5=12.502, Q1.4=9.044,
# Q1.3=8.512, pow2=7.182 um^2 -> savings vs exact.
import matplotlib.pyplot as plt
AREA_SAVE = {"exact": 0.0, "q1b5": 17.54, "q1b4": 40.35, "q1b3": 43.86, "pow2": 52.63}
LABEL = {"exact": "exact GF4 (Q1.7)", "q1b5": "Q1.5", "q1b4": "Q1.4", "q1b3": "Q1.3", "pow2": "pow2"}
sm = "fp16"
pts = [(AREA_SAVE[n], d2, LABEL[n]) for (s, n, p, d1, d2) in rows if s == sm and n in AREA_SAVE]
pts.sort()
xs, ys, names = zip(*pts)
fig, ax = plt.subplots(figsize=(6.2, 4.2))
ax.plot(xs, ys, "-o", color="#1f77b4", lw=1.5, ms=7)
for x, y, n in pts:
    ax.annotate(n, (x, y), textcoords="offset points", xytext=(6, 8 if y < 5 else -12), fontsize=9)
if "Q1.4" in names:
    i = names.index("Q1.4")
    ax.scatter([xs[i]], [ys[i]], s=180, facecolors="none", edgecolors="#d62728", lw=2)
ax.axhline(0, color="grey", lw=0.6, ls=":")
ax.set_xlabel("Decode-ROM area savings vs. exact GF4 (%)")
ax.set_ylabel(r"$\Delta$PPL vs. exact GF4")
ax.set_title(f"GF4 codebook regularity: accuracy vs. decode area\n({MODEL}, scale={sm})")
ax.grid(True, alpha=0.3); fig.tight_layout()
fig.savefig("gf4_regularity_pareto.png", dpi=160); plt.show()
print("wrote gf4_regularity_pareto.png")